# Identifying Underserved Healthcare Counties Using Alternative Data

This notebook walks through the full analysis pipeline for identifying North Carolina counties with excess emergency department (ED) burden using publicly available alternative data sources.

**Key contributions:**
1. Demonstrates that publicly available data (CMS, CDC SVI, CDC PLACES, HRSA) can explain ~61% of county-level ED volume variation
2. Shows that the same data *cannot* predict ED utilization rates (R² < 0), implying local healthcare system factors dominate
3. Uses model residuals to identify counties with excess ED burden — a data-driven approach to mapping underserved areas
4. Pipeline is generalizable to any US state with county-level ED data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = Path('../data/final/merged_county_data.csv')
print(f'Data path: {DATA_PATH.resolve()}')

## 1. Data Overview

The merged dataset combines 5 nationally available data sources at the county level. North Carolina has 100 counties.

In [ ]:
df = pd.read_csv(DATA_PATH)
for col in df.columns:
    if col not in ['County', 'Region']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Counties: {df.shape[0]}')
print(f'Columns:  {df.shape[1]}')
print(f'\nRegion distribution:')
print(df['Region'].value_counts().to_string())
print(f'\nCounties with 0 ED visits (no hospital): {(df["Total_ED"] == 0).sum()}')
print(f'Counties with ED facilities: {(df["Total_ED"] > 0).sum()}')

## 2. Target Variable: ED Utilization

**Important**: The ED data counts visits *at hospitals in a county*, not visits *by residents*. Counties with regional medical centers can show rates > 1,000 per 1,000 population because they serve neighboring counties without hospitals.

In [ ]:
df_model = df[df['Total_ED'] > 0].copy()
df_no_ed = df[df['Total_ED'] == 0].copy()

df_model['ED_rate'] = df_model['Total_ED'] / df_model['E_TOTPOP'] * 1000

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Total ED distribution
axes[0].hist(df_model['Total_ED'], bins=25, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Total ED Visits')
axes[0].set_ylabel('Number of Counties')
axes[0].set_title('Distribution of Total ED Visits')

# ED rate distribution
axes[1].hist(df_model['ED_rate'], bins=25, color='#FF5722', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('ED Visits per 1,000 Population')
axes[1].set_ylabel('Number of Counties')
axes[1].set_title('Distribution of ED Rate')

# Log(ED) distribution
axes[2].hist(np.log(df_model['Total_ED']), bins=25, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[2].set_xlabel('log(Total ED Visits)')
axes[2].set_ylabel('Number of Counties')
axes[2].set_title('Distribution of log(Total ED)')

plt.tight_layout()
plt.show()

print(f'\nED Rate Summary (82 counties with hospitals):')
print(df_model['ED_rate'].describe().to_string())
print(f'\nCounties with ED rate > 1000/1000 (serve neighboring counties):')
print(df_model[df_model['ED_rate'] > 1000][['County', 'Region', 'E_TOTPOP', 'Total_ED', 'ED_rate']].to_string(index=False))

## 3. Healthcare Deserts

18 counties have no ED facility at all. These are automatically classified as underserved.

In [ ]:
desert_cols = ['County', 'Region', 'E_TOTPOP', 'EP_UNINSUR', 'EP_POV150',
               'HPSA_PrimaryCare_2023', 'MDs_PrimaryCare_perCap_2021']
print('Counties without ED facilities (healthcare deserts):')
print(df_no_ed[desert_cols].sort_values('E_TOTPOP', ascending=False).to_string(index=False))
print(f'\nTotal population in healthcare deserts: {df_no_ed["E_TOTPOP"].sum():,.0f}')

## 4. Feature Engineering

We engineer features from the raw data to create rate-based and log-transformed predictors. All features come from nationally available sources.

In [ ]:
# Feature engineering
df_model['log_Pop'] = np.log(df_model['E_TOTPOP'].clip(lower=1))
df_model['log_MA'] = np.log1p(df_model['AVG_MA_Enrollments_2023_FY'])
df_model['log_MC'] = np.log1p(df_model['AVG_MC_Enrollees'])
df_model['pub_insured_pct'] = (
    (df_model['AVG_MA_Enrollments_2023_FY'] + df_model['AVG_MC_Enrollees'])
    / df_model['E_TOTPOP'].clip(lower=1) * 100
)
df_model['MDs_per_10k'] = df_model['MDs_All_2021'] / df_model['E_TOTPOP'].clip(lower=1) * 10000
df_model['PCP_per_10k'] = df_model['MDs_PrimaryCare_perCap_2021'] / df_model['E_TOTPOP'].clip(lower=1) * 10000
df_model['log_Total_ED'] = np.log(df_model['Total_ED'].clip(lower=1))

# Define feature groups
feature_groups = {
    'Insurance (CMS)': ['pub_insured_pct', 'log_MA', 'log_MC'],
    'SVI (CDC)': ['RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4'],
    'Demographics': ['EP_POV150', 'EP_UNEMP', 'EP_UNINSUR', 'EP_AGE65', 'EP_AGE17',
                     'EP_DISABL', 'EP_MINRTY', 'EP_MOBILE', 'EP_NOVEH', 'EP_NOINT'],
    'Infrastructure (HRSA)': ['HPSA_PrimaryCare_2023', 'HPSA_Dental_2023',
                              'HPSA_Mental_2023', 'MDs_per_10k', 'PCP_per_10k'],
    'Disease (PLACES)': [c for c in df_model.columns if 'age-adjusted_prevalence' in c],
    'Rurality': ['RUCC_2013', 'UIC_2013'],
}

all_features = ['log_Pop']
for group_cols in feature_groups.values():
    all_features.extend([c for c in group_cols if c in df_model.columns])

print(f'Total engineered features: {len(all_features)}')
for group, cols in feature_groups.items():
    avail = [c for c in cols if c in df_model.columns]
    print(f'  {group}: {len(avail)} features')
print(f'  Population (offset): 1 feature')

## 5. Model Comparison

Run the full pipeline and display results.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.model.pipeline import run_pipeline

results = run_pipeline()

## 6. Results Visualization

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path('../results')

print('Model Comparison (Model A vs Model B):')
display(Image(filename=str(fig_dir / 'fig_model_comparison.png'), width=900))

In [ ]:
print('Predicted vs Actual (Best Model):')
display(Image(filename=str(fig_dir / 'fig_pred_vs_actual_model_a.png'), width=500))

In [ ]:
print('Feature Importance by Data Source:')
display(Image(filename=str(fig_dir / 'fig_data_source_importance.png'), width=700))
print()
print('Top 20 Individual Features:')
display(Image(filename=str(fig_dir / 'fig_feature_importance.png'), width=700))

In [ ]:
print('Excess ED Burden by County:')
display(Image(filename=str(fig_dir / 'fig_excess_burden.png'), width=700))

In [ ]:
print('Parsimony Analysis (How few features are enough?):')
display(Image(filename=str(fig_dir / 'fig_parsimony.png'), width=600))

## 7. Excess Burden Table

Counties ranked by excess ratio (observed / expected ED visits). Values > 1.0 indicate more ED visits than the model predicts from demographics and infrastructure alone.

In [ ]:
excess = results['excess_burden'].copy()
display_cols = ['County', 'Region', 'E_TOTPOP', 'Total_ED', 'ED_rate_per_1000',
                'Excess_ratio', 'Classification']
avail_cols = [c for c in display_cols if c in excess.columns]

print('Top 15 counties by excess ratio:')
print(excess.head(15)[avail_cols].to_string(index=False))
print()
print('Bottom 5 counties by excess ratio:')
print(excess.tail(5)[avail_cols].to_string(index=False))

## 8. Full Underserved Map

Combining excess-burden counties with healthcare deserts gives a complete picture of underserved NC counties.

In [ ]:
underserved = results['underserved_map']
flagged = underserved[underserved['Classification'].isin(['Excess Burden', 'No ED Facility'])]

print(f'Total underserved counties: {len(flagged)} of {len(underserved)}')
print()
print('Classification breakdown:')
print(underserved['Classification'].value_counts().to_string())
print()
print('Underserved counties:')
flag_cols = ['County', 'Region', 'E_TOTPOP', 'Classification']
avail = [c for c in flag_cols if c in flagged.columns]
print(flagged.sort_values('Classification')[avail].to_string(index=False))

## 9. Discussion

### Key Takeaways

1. **Alternative data works for volume prediction (R-squared = 0.61)** but not for rate prediction (R-squared < 0). This means population and a few structural factors explain *how many* ED visits occur, but *not the rate* relative to population.

2. **The failure of rate prediction is the finding.** It implies that unmeasured local factors — hospital catchment areas, availability of urgent care and primary care clinics, transportation infrastructure, cultural healthcare-seeking patterns — are the dominant drivers of ED utilization rates.

3. **The excess burden approach provides actionable intelligence.** Even with moderate R-squared, the residuals identify counties where observed ED utilization systematically exceeds what demographics predict. These are candidates for targeted interventions.

4. **18% of NC counties are healthcare deserts.** These counties have no ED facility, meaning residents must travel to neighboring counties for emergency care. These are categorically underserved.

5. **The pipeline is generalizable.** All predictor features come from nationally available data sources. Only the target variable (ED visits) requires state-specific data.

### Limitations

- ED data is facility-based, not residence-based. Counties with regional medical centers may show artificially high rates.
- Small sample size (82 counties for modeling) limits model complexity.
- Cross-sectional analysis; temporal trends in ED utilization are not captured in the model.
- Some features (RUCC, HPSA designations) may be endogenous to the outcome.